# M36 lab solution: parallel retrieval

**Brief.** Run the sub-questions of a decomposed query as parallel agents and stream partial answers, then report the latency saved.

The six `multi_hop` questions of the Larkspur golden set need evidence from two or more handbook files. For each one:

1. A decomposer splits the question into sub-questions (structured output).
2. Each sub-question goes to its own retrieval agent. In the parallel graph the fan-out is `Send`, as in notebook `05a_parallel_agents.ipynb`; in the sequential graph one node answers the same sub-questions one after another.
3. A synthesizer joins the partial answers into one reply.

Both graphs get the same sub-questions, so the timing difference comes only from running the agents at once. Partial answers are printed as they arrive with `stream_mode="updates"`.

## Setup

In [ ]:
import operator
import statistics
import sys
import time
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import is_hit, load_handbook, load_jsonl, tfidf_retriever  # noqa: E402

DATA = Path("../data/larkspur")
llm = get_experientiallabs_llm()
MODEL = llm.model_name

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for page in load_handbook(DATA / "handbook") for c in splitter.split_text(page["text"])]
search = tfidf_retriever(chunks, k=3)
golden = load_jsonl(DATA / "golden.jsonl")
multi_hop = [g for g in golden if g["type"] == "multi_hop"]
print("model:", MODEL, "|", len(chunks), "chunks |", len(multi_hop), "multi-hop questions")

## Decompose

In [ ]:
class SubQuestions(BaseModel):
    sub_questions: list[str] = Field(
        description="Two to four self-contained questions, each answerable from one section of a cloud host's handbook."
    )


decomposer = llm.with_structured_output(SubQuestions)


def decompose(question: str) -> list[str]:
    out = decomposer.invoke([
        SystemMessage("Split the customer question into the smaller questions you would look up separately."),
        HumanMessage(question),
    ])
    return out.sub_questions[:4]


print(decompose(multi_hop[0]["question"]))

## The retrieval agent and the synthesizer

In [ ]:
def answer_sub_question(sub_question: str) -> dict:
    passages = search(sub_question)
    reply = llm.invoke([
        SystemMessage("Answer from the passages only, in one or two sentences. If they do not answer, say so.\n\n"
                      + "\n---\n".join(passages)),
        HumanMessage(sub_question),
    ]).content
    return {"sub_question": sub_question, "answer": reply, "passages": passages}


def synthesize_answer(question: str, partials: list[dict]) -> str:
    notes = "\n".join(f"- {p['sub_question']}: {p['answer']}" for p in partials)
    return llm.invoke([
        SystemMessage("Combine the notes into one short answer to the customer's question. Use only the notes."),
        HumanMessage(f"Question: {question}\n\nNotes:\n{notes}"),
    ]).content


class State(TypedDict):
    question: str
    sub_questions: list[str]
    partials: Annotated[list[dict], operator.add]
    answer: str


class AgentInput(TypedDict):
    sub_question: str


def agent(state: AgentInput) -> dict:
    return {"partials": [answer_sub_question(state["sub_question"])]}


def synthesize(state: State) -> dict:
    return {"answer": synthesize_answer(state["question"], state["partials"])}

## Two graphs over the same sub-questions

In [ ]:
def fan_out(state: State) -> list[Send]:
    return [Send("agent", {"sub_question": q}) for q in state["sub_questions"]]


par = StateGraph(State)
par.add_node("agent", agent)
par.add_node("synthesize", synthesize)
par.add_conditional_edges(START, fan_out, ["agent"])
par.add_edge("agent", "synthesize")
par.add_edge("synthesize", END)
parallel_graph = par.compile()


def all_in_turn(state: State) -> dict:
    return {"partials": [answer_sub_question(q) for q in state["sub_questions"]]}


seq = StateGraph(State)
seq.add_node("agents_in_turn", all_in_turn)
seq.add_node("synthesize", synthesize)
seq.add_edge(START, "agents_in_turn")
seq.add_edge("agents_in_turn", "synthesize")
seq.add_edge("synthesize", END)
sequential_graph = seq.compile()

print(parallel_graph.get_graph().draw_mermaid())

## Streaming partial answers

`stream_mode="updates"` yields one update per finished node. Each agent's partial answer prints as soon as that agent ends; the synthesizer's answer comes last.

In [ ]:
def run_streaming(graph, question: str, sub_questions: list[str], show: bool = True) -> tuple[dict, float]:
    state = {"question": question, "sub_questions": sub_questions, "partials": [], "answer": ""}
    final = dict(state)
    start = time.perf_counter()
    for update in graph.stream(state, stream_mode="updates"):
        for node, delta in update.items():
            if not delta:
                continue
            for p in delta.get("partials", []):
                final["partials"] = final["partials"] + [p]
                if show:
                    print(f"  [{time.perf_counter() - start:5.2f}s] {node}: {p['sub_question']} -> {p['answer'][:120]}")
            if "answer" in delta:
                final["answer"] = delta["answer"]
                if show:
                    print(f"  [{time.perf_counter() - start:5.2f}s] {node}: {delta['answer'][:200]}")
    return final, time.perf_counter() - start


g = multi_hop[0]
subs = decompose(g["question"])
print(g["question"])
_ = run_streaming(parallel_graph, g["question"], subs)

## Sequential against parallel on the six multi-hop questions

Each question is decomposed once and both graphs answer the same sub-questions. The order of the two runs alternates per question so neither always runs second against a warm connection. An evidence hit is counted when any passage an agent read contains one of the question's evidence spans (`lab_kit.is_hit`).

In [ ]:
rows = []
for i, g in enumerate(multi_hop):
    subs = decompose(g["question"])
    order = [("parallel", parallel_graph), ("sequential", sequential_graph)]
    if i % 2:
        order.reverse()
    for name, graph in order:
        final, seconds = run_streaming(graph, g["question"], subs, show=False)
        read = [c for p in final["partials"] for c in p["passages"]]
        spans_found = sum(any(is_hit(c, [s]) for c in read) for s in g["evidence_spans"])
        rows.append({"id": g["id"], "graph": name, "sub_questions": len(subs), "seconds": round(seconds, 2),
                     "spans_found": f"{spans_found}/{len(g['evidence_spans'])}", "answer": final["answer"]})
    print(g["id"], subs)

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
wide = df.pivot(index="id", columns="graph", values="seconds")
wide["saved_s"] = wide["sequential"] - wide["parallel"]
wide["sub_questions"] = df.groupby("id")["sub_questions"].first()
print("model:", MODEL, "| run:", time.strftime("%Y-%m-%d"))
print(wide.to_markdown())
print()
print(f"median sequential {statistics.median(wide['sequential']):.2f}s, "
      f"median parallel {statistics.median(wide['parallel']):.2f}s, "
      f"median saved {statistics.median(wide['saved_s']):.2f}s")

In [ ]:
for g in multi_hop:
    r = df[(df["id"] == g["id"]) & (df["graph"] == "parallel")].iloc[0]
    print(f"{g['id']} {g['question']}\n  answer: {r['answer']}\n  reference: {g['reference_answer']}\n  spans found: {r['spans_found']}\n")

## What this shows

- LangGraph runs the `Send` branches of one superstep at once, and the step ends when the slowest branch ends. The parallel time is roughly one agent's time plus synthesis, while the sequential time grows with every sub-question.
- The saving grows with the number of sub-questions, so it is largest on the questions the decomposer split three or four ways.
- Parallelism changes latency, not evidence: both graphs read the same passages. If a span is missing, the fix is in decomposition or retrieval, not in the fan-out.